Installation of transformers and captum



In [ ]:
!pip install transformers
!pip install captum
!pip install pandas
!pip install seaborn
!pip install transformers-interpret
!pip install gspread oauth2client
!pip install shap
!pip install lime
!pip install openpyxl


  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
captum 0.8.0 requires numpy<2.0, but you have numpy 2.0.2 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

Import of all packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from transformers_interpret import SequenceClassificationExplainer

from captum.attr import GradientShap

import shap


import ast
import re
from lime.lime_text import LimeTextExplainer


Use of the RoBERTa base openai detector model and tokenizer

In [ ]:
# Setup
tokenizer = AutoTokenizer.from_pretrained("openai-community/roberta-base-openai-detector")
model = AutoModelForSequenceClassification.from_pretrained("openai-community/roberta-base-openai-detector")
model.eval()

def forward_embeds(embeds):
    return model(inputs_embeds=embeds).logits


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


WhiteBox (GradientSHAP)

In [ ]:

# Input
text = "There are many recipes for a delicious fruit salad my favourite fruit salad contains oranges, cherries, some banana and a few kiwis and 1 mango. The mix of those exotic fruits make the salad very delightful and makes as a good desert after a nice dinner in the evening."
inputs = tokenizer(text, return_tensors="pt")
input_ids = inputs["input_ids"]

# 1. Prediction & Confidence
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)
    pred_class = probs.argmax(dim=-1).item()
    confidence = probs[0, pred_class].item()

# 2. Attribution Setup
embeddings = model.get_input_embeddings()(input_ids).detach()
embeddings.requires_grad_()
baseline = torch.zeros_like(embeddings)

gs = GradientShap(forward_embeds)

# 3. Calculate Attributions (Targeting the predicted class)
attributions = gs.attribute(
    embeddings,
    baselines=baseline,
    n_samples=50,
    stdevs=0.001,
    target=pred_class
)

# 4. Processing Results
token_attributions = attributions.sum(dim=-1).squeeze(0)
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))
labels = {1: "Real (Human)", 0: "Fake (AI)"}

# Output
print(f"Predicted class: {pred_class} ({labels[pred_class]})")
print(f"Confidence: {confidence:.4f}\n")
print("Token-level attributions (Explaining the prediction):")
for tok, score in zip(tokens, token_attributions):
    print(f"{tok:>12s} : {score.item(): .4f}")

WhiteBox (transformers-interpret)

In [ ]:
cls_explainer = SequenceClassificationExplainer(model, tokenizer)
cls_explainer(text)

BlackBox (LIME)

In [ ]:
# ============================================================
# LIME Blackbox XAI → direkt in bestehende Excel schreiben
#
# Gleicher Input wie GradientShap:
#   inputs = tokenizer(text, return_tensors="pt")
# LIME bekommt dieselben token_ids → gleiche Prediction garantiert.
#
# Colab Setup:
#   !pip install lime transformers torch openpyxl pandas
#   Runtime → Change runtime type → T4 GPU
# ============================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from lime.lime_text import LimeTextExplainer

# ── 1. Model Setup (identisch zu GradientShap-Notebook) ──────
MODEL_NAME = "openai-community/roberta-base-openai-detector"

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
model.to(device)

LABELS = list(model.config.id2label.values())
print(f"Labels: {LABELS}")

# ── 2. Predict-Funktion ──────────────────────────────────────
def predict_for_text(text: str):
    """
    Exakt wie GradientShap:
      inputs = tokenizer(text, return_tensors="pt")
    Gibt (probs, input_ids, tokens) zurück.
    """
    inputs    = tokenizer(text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        logits = model(**inputs).logits
        probs  = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))
    return probs, input_ids, tokens

def predict_proba_from_ids(id_lists: list) -> np.ndarray:
    """
    LIME übergibt perturbierte Token-Sequenzen als Listen von IDs.
    Wir bauen daraus direkt input_ids Tensoren → kein Tokenizer-Aufruf nötig.
    """
    all_probs = []
    BATCH = 32
    for i in range(0, len(id_lists), BATCH):
        batch = id_lists[i : i + BATCH]

        # Padding auf gleiche Länge
        max_len   = max(len(ids) for ids in batch)
        pad_id    = tokenizer.pad_token_id
        padded    = [ids + [pad_id] * (max_len - len(ids)) for ids in batch]
        masks     = [[1] * len(ids) + [0] * (max_len - len(ids)) for ids in batch]

        input_ids      = torch.tensor(padded, dtype=torch.long).to(device)
        attention_mask = torch.tensor(masks,  dtype=torch.long).to(device)

        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

# ── 3. LIME Explanation ──────────────────────────────────────
def explain_lime(text: str, num_samples: int = 150) -> str:
    """
    1. Tokenize genau wie GradientShap
    2. LIME maskiert Token-IDs direkt (kein String-Splitting-Problem)
    3. predict_proba bekommt ID-Listen → identischer Modell-Input
    """
    probs, input_ids, tokens = predict_for_text(text)

    # Special tokens (<s>, </s>) rausfiltern für LIME
    ids_list      = input_ids.squeeze(0).tolist()
    special_ids   = {tokenizer.bos_token_id, tokenizer.eos_token_id,
                     tokenizer.pad_token_id}
    content_idxs  = [i for i, id_ in enumerate(ids_list) if id_ not in special_ids]
    content_ids   = [ids_list[i] for i in content_idxs]
    content_tokens = [tokens[i] for i in content_idxs]

    if len(content_ids) < 3:
        return "Text too short for LIME explanation."

    pred_idx   = int(np.argmax(probs))
    pred_label = LABELS[pred_idx]
    confidence = float(probs[pred_idx])

    # LIME-Perturbation: wir arbeiten mit Token-Index-Strings
    # LIME ersetzt Einträge durch '' → wir filtern diese beim Rekonstruieren
    token_index_strings = [str(i) for i in range(len(content_ids))]
    token_string        = ' '.join(token_index_strings)

    def predict_from_lime_string(lime_strings: list) -> np.ndarray:
        """
        LIME übergibt Strings wie '0 1 2 4 5' (Token-Indizes, maskierte fehlen).
        Wir rekonstruieren daraus ID-Listen und rufen predict_proba_from_ids auf.
        """
        id_lists = []
        for s in lime_strings:
            present_idxs = [int(x) for x in s.split() if x.isdigit()]
            ids = [content_ids[i] for i in present_idxs if i < len(content_ids)]
            # Special tokens wieder hinzufügen
            full_ids = [tokenizer.bos_token_id] + ids + [tokenizer.eos_token_id]
            id_lists.append(full_ids)
        return predict_proba_from_ids(id_lists)

    try:
        lime_explainer = LimeTextExplainer(
            class_names=LABELS,
            split_expression=' ',
            bow=False
        )
        explanation = lime_explainer.explain_instance(
            token_string,
            predict_from_lime_string,
            num_features=len(content_ids),
            num_samples=num_samples,
            labels=[pred_idx]
        )
        # LIME gibt Index-Strings zurück → zurück zu Token-Names mappen
        attr_by_idx = {k: v for k, v in explanation.as_list(label=pred_idx)}
    except Exception as e:
        return f"LIME Error: {e}"

    lines = [
        f"Predicted class: {pred_idx} ({pred_label})",
        f"Confidence: {confidence:.4f}",
        "",
        "Token-level attributions (LIME Blackbox):",
        "(sorted by position in text; score = influence on predicted class)"
    ]
    for i, token in enumerate(content_tokens):
        score        = attr_by_idx.get(str(i), 0.0)
        display_name = token.replace('Ġ', ' ').strip()
        lines.append(f"  {display_name:>20} : {score:+.4f}")

    return "\n".join(lines)

# ── 4. Excel einlesen ────────────────────────────────────────
EXCEL_PATH  = "XAi_Scores.xlsx"
OUTPUT_PATH = "XAi_Scores_with_LIME_v6.xlsx"

df = pd.read_excel(EXCEL_PATH, sheet_name=0, header=None)
print(f"\nExcel geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

question_rows = []
for i in range(df.shape[0]):
    val = df.iloc[i, 0]
    if isinstance(val, str) and val.strip().startswith("Question"):
        question_rows.append(i)

print(f"Gefundene Blöcke: {len(question_rows)}")

# ── 5. LIME berechnen und eintragen ─────────────────────────
COL_START  = 1
COL_END    = 7
texts_done = 0
errors     = 0

for idx, block_start in enumerate(question_rows):
    block_end = question_rows[idx + 1] if idx + 1 < len(question_rows) else df.shape[0]

    question_label = df.iloc[block_start, 0]
    print(f"\n{'='*55}")
    print(f"Block: '{question_label}'")

    text_row = None
    lime_row = None
    for r in range(block_start, block_end):
        val = str(df.iloc[r, 0]).strip()
        if r == block_start:
            continue
        if val == "BlackBox":
            lime_row = r
        elif val not in ("WhiteBox", "GradientShap", "BlackBox", "nan", ""):
            if text_row is None and isinstance(df.iloc[r, 0], str):
                text_row = r

    if text_row is None:
        text_row = block_start + 1

    if lime_row is None:
        print("  Keine BlackBox-Zeile, übersprungen")
        continue

    if not isinstance(df.iloc[text_row, 0], str) or len(str(df.iloc[text_row, 0]).strip()) < 2:
        print("  Kein Text, übersprungen")
        continue

    for col in range(COL_START, COL_END):
        text_cell = df.iloc[text_row, col]

        if not isinstance(text_cell, str) or len(text_cell.strip()) < 10:
            print(f"  Spalte {col}: leer, übersprungen")
            continue

        existing = df.iloc[lime_row, col]
        if isinstance(existing, str) and len(existing.strip()) > 5:
            print(f"  Spalte {col}: bereits befüllt, übersprungen")
            continue

        _, input_ids, _ = predict_for_text(text_cell)
        n_tokens = input_ids.shape[1]
        print(f"  Spalte {col}: läuft... ({n_tokens} Tokens)", end="", flush=True)

        lime_result = explain_lime(text_cell)

        if "Error" in lime_result:
            errors += 1
            print(f" → FEHLER: {lime_result[:80]}")
        else:
            texts_done += 1
            print(f" → {lime_result.split(chr(10))[0]}")

        df.iloc[lime_row, col] = lime_result

        with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="Sheet1", index=False, header=False)

print(f"\n{'='*55}")
print(f"Fertig! {texts_done} OK, {errors} Fehler.")
print(f"Gespeichert: {OUTPUT_PATH}")

Device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: openai-community/roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Labels: ['Fake', 'Real']

Excel geladen: 79 Zeilen, 7 Spalten
Gefundene Blöcke: 16

Block: 'Question_1'
  Spalte 1: bereits befüllt, übersprungen
  Spalte 2: bereits befüllt, übersprungen
  Spalte 3: bereits befüllt, übersprungen
  Spalte 4: bereits befüllt, übersprungen
  Spalte 5: bereits befüllt, übersprungen
  Spalte 6: bereits befüllt, übersprungen

Block: 'Question_2'
  Spalte 1: bereits befüllt, übersprungen
  Spalte 2: bereits befüllt, übersprungen
  Spalte 3: bereits befüllt, übersprungen
  Spalte 4: bereits befüllt, übersprungen
  Spalte 5: bereits befüllt, übersprungen
  Spalte 6: bereits befüllt, übersprungen

Block: 'Question_3'
  Spalte 1: bereits befüllt, übersprungen
  Spalte 2: bereits befüllt, übersprungen
  Spalte 3: bereits befüllt, übersprungen
  Spalte 4: bereits befüllt, übersprungen
  Spalte 5: bereits befüllt, übersprungen
  Spalte 6: bereits befüllt, übersprungen

Block: 'Question_4'
  Spalte 1: bereits befüllt, übersprungen
  Spalte 2: bereits befüllt, übersp

Analysis

In [ ]:
# ============================================================
# XAI Comparison Analysis
# Methods: WhiteBox (TransformersInterpret), GradientShap, LIME
# Data: 15 questions × 6 texts (3 AI / 3 Human) = 90 texts
#
# Analyses:
#   1. Top-N / Bottom-N words per method  → CSV export
#   2. Token overlap between methods
#   3. Confidence: AI vs Human texts
#   4. Sign agreement between methods
#   5. Score magnitude per method
#   6. Attribution spread (concentration ratio)
#
# !pip install pandas openpyxl matplotlib seaborn scipy numpy
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, ast, os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
})

EXCEL_PATH = "XAi_Scores4_with_LIME.xlsx"   # <-- adjust if needed
OUT_DIR    = "xai_results"
os.makedirs(OUT_DIR, exist_ok=True)


# ════════════════════════════════════════════════════════════
# PARSERS  (tested — 0 errors across all 90 cells)
# ════════════════════════════════════════════════════════════

def parse_whitebox(cell: str) -> dict:
    """
    WhiteBox stores attributions as a Python list of (token, score) tuples.
    We strip RoBERTa subword markers (Ġ = space prefix, Ċ = newline).
    Special tokens <s> and </s> are excluded.
    """
    try:
        pairs = ast.literal_eval(cell.strip())
        out = {}
        for token, score in pairs:
            clean = token.replace('Ġ', '').replace('Ċ', '').strip()
            if clean and clean not in ('<s>', '</s>', '<pad>'):
                out[clean] = float(score)
        return out
    except Exception:
        return {}

def parse_gradientshap(cell: str) -> tuple:
    """
    GradientShap stores: header lines + '  token : score' lines.
    Returns (dict of token→score, confidence float, predicted class int).
    """
    out = {}
    confidence = None
    pred_class = None
    pattern = r'^\s*(.+?)\s*:\s*([+-]?\d+\.\d+)\s*$'
    for line in cell.strip().split('\n'):
        if 'Confidence:' in line:
            m = re.search(r'Confidence:\s*([\d.]+)', line)
            if m:
                confidence = float(m.group(1))
        elif 'Predicted class:' in line:
            pred_class = 1 if 'Real' in line else 0
        else:
            m = re.match(pattern, line)
            if m:
                token = m.group(1).strip().replace('Ġ', '').replace('Ċ', '')
                if token and token not in ('<s>', '</s>', '<pad>'):
                    out[token] = float(m.group(2))
    return out, confidence, pred_class

def parse_lime(cell: str) -> tuple:
    """
    LIME stores: header lines + '  word : score' lines.
    Returns (dict of word→score, confidence float, predicted class int).
    LIME works at word level (no subword tokens).
    """
    out = {}
    confidence = None
    pred_class = None
    pattern = r'^\s*(.+?)\s*:\s*([+-]?\d+\.\d+)\s*$'
    for line in cell.strip().split('\n'):
        if 'Confidence:' in line:
            m = re.search(r'Confidence:\s*([\d.]+)', line)
            if m:
                confidence = float(m.group(1))
        elif 'Predicted class:' in line:
            pred_class = 1 if 'Real' in line else 0
        elif 'sorted by' in line or 'Word-level' in line:
            continue
        else:
            m = re.match(pattern, line)
            if m:
                word = m.group(1).strip()
                if word:
                    out[word] = float(m.group(2))
    return out, confidence, pred_class

def normalize_keys(d: dict) -> dict:
    """Lowercase + remove Ġ prefix so WB/GS tokens match LIME words."""
    return {k.replace('Ġ', '').lower(): v for k, v in d.items()}


# ════════════════════════════════════════════════════════════
# LOAD DATA
# ════════════════════════════════════════════════════════════

df_raw = pd.read_excel(EXCEL_PATH, sheet_name=0, header=None)

question_rows = [i for i in range(df_raw.shape[0])
                 if isinstance(df_raw.iloc[i, 0], str)
                 and df_raw.iloc[i, 0].strip().startswith('Question')]

COL_NAMES = ['Gpt_Answer_1', 'Gpt_Answer_2', 'Gpt_Answer_3',
             'Human_Answer_1', 'Human_Answer_2', 'Human_Answer_3']

records = []
for qr in question_rows:
    q_label = str(df_raw.iloc[qr,   0]).strip()
    q_text  = str(df_raw.iloc[qr+1, 0]).strip()
    for col_idx, col_name in enumerate(COL_NAMES, start=1):
        wb_val = df_raw.iloc[qr+2, col_idx]
        gs_val = df_raw.iloc[qr+3, col_idx]
        bb_val = df_raw.iloc[qr+4, col_idx]
        if not isinstance(wb_val, str):
            continue
        wb                       = parse_whitebox(wb_val)
        gs, gs_conf, gs_pred     = parse_gradientshap(gs_val)
        lime, lime_conf, lime_pred = parse_lime(bb_val)
        records.append({
            'question':    q_label,
            'q_text':      q_text,
            'answer_type': col_name,
            'is_ai':       col_idx <= 3,
            'full_text':   str(df_raw.iloc[qr+1, col_idx]),
            'wb':          wb,
            'gs':          gs,
            'lime':        lime,
            'gs_conf':     gs_conf,
            'lime_conf':   lime_conf,
            'gs_pred':     gs_pred,
            'lime_pred':   lime_pred,
        })

data = pd.DataFrame(records)
print(f"Loaded {len(data)} texts  |  AI: {data['is_ai'].sum()}  Human: {(~data['is_ai']).sum()}")


# ════════════════════════════════════════════════════════════
# ANALYSIS 1 — TOP-N / BOTTOM-N WORDS PER METHOD
#
# What: For each text and each method, rank tokens by their
# attribution score. Top = tokens that push toward the
# predicted class. Bottom = tokens pushing against it.
#
# Why useful: Lets us directly read which words each method
# considers most influential — and compare whether the three
# methods highlight the same words.
# ════════════════════════════════════════════════════════════

N_TOPBOT = 10

def get_top_n(scores: dict, n: int, mode: str) -> list:
    if not scores:
        return []
    if mode == 'top':
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
    else:
        return sorted(scores.items(), key=lambda x: x[1])[:n]

# ── Save all top/bottom to CSV ───────────────────────────────
topbot_rows = []
for _, row in data.iterrows():
    for method, key in [('WhiteBox', 'wb'), ('GradientShap', 'gs'), ('LIME', 'lime')]:
        scores = row[key]
        top = get_top_n(scores, N_TOPBOT, 'top')
        bot = get_top_n(scores, N_TOPBOT, 'bottom')
        for rank, (word, score) in enumerate(top, 1):
            topbot_rows.append({'question': row['question'], 'q_text': row['q_text'],
                'answer_type': row['answer_type'], 'is_ai': row['is_ai'],
                'method': method, 'rank_type': 'top',
                'rank': rank, 'word': word, 'score': round(score, 5)})
        for rank, (word, score) in enumerate(bot, 1):
            topbot_rows.append({'question': row['question'], 'q_text': row['q_text'],
                'answer_type': row['answer_type'], 'is_ai': row['is_ai'],
                'method': method, 'rank_type': 'bottom',
                'rank': rank, 'word': word, 'score': round(score, 5)})

topbot_df = pd.DataFrame(topbot_rows)
topbot_df.to_csv(f'{OUT_DIR}/top_bottom_words_all.csv', index=False)
print(f"Saved: {OUT_DIR}/top_bottom_words_all.csv  ({len(topbot_df)} rows)")

# ── Plot: one representative example per question (Gpt_Answer_1) ─
# Pick 6 examples to show (every other question for readability)
sample_indices = data[data['answer_type'] == 'Gpt_Answer_1'].index[:6].tolist()
N_SHOW = 5  # top/bottom 5 in the plot

fig, axes = plt.subplots(len(sample_indices), 3, figsize=(14, len(sample_indices) * 2.8))
fig.subplots_adjust(hspace=0.6, wspace=0.45)
METHOD_COLORS = {'WhiteBox': '#2c6fad', 'GradientShap': '#2e8b57', 'LIME': '#b03a2e'}
METHODS = [('WhiteBox', 'wb'), ('GradientShap', 'gs'), ('LIME', 'lime')]

for row_i, idx in enumerate(sample_indices):
    row = data.loc[idx]
    for col_i, (method, key) in enumerate(METHODS):
        ax = axes[row_i][col_i]
        scores = row[key]
        top = get_top_n(scores, N_SHOW, 'top')
        bot = get_top_n(scores, N_SHOW, 'bottom')
        items = bot + top
        words = [w for w, _ in items]
        vals  = [s for _, s in items]
        colors = ['#c0392b' if v < 0 else '#1a7a4a' for v in vals]
        ax.barh(range(len(words)), vals, color=colors, height=0.6)
        ax.set_yticks(range(len(words)))
        ax.set_yticklabels(words, fontsize=8)
        ax.axvline(0, color='#333', linewidth=0.6)
        ax.set_xlabel('Score', fontsize=8)
        if row_i == 0:
            ax.set_title(method, fontsize=10, color=METHOD_COLORS[method], fontweight='bold')
        # Question label on left
        if col_i == 0:
            short_q = row['q_text'][:35] + '…' if len(row['q_text']) > 35 else row['q_text']
            label   = f"{row['question']}\n\"{short_q}\"\n({'AI' if row['is_ai'] else 'Human'})"
            ax.set_ylabel(label, fontsize=7.5, rotation=0, labelpad=120, va='center')

plt.savefig(f'{OUT_DIR}/analysis_1_top_bottom.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/analysis_1_top_bottom.png', bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_DIR}/analysis_1_top_bottom.png")


# ════════════════════════════════════════════════════════════
# ANALYSIS 2 — TOKEN OVERLAP BETWEEN METHODS
#
# What: Take the Top-10 most influential tokens (by |score|)
# for each method. Count how many appear in all method pairs.
#
# Why useful: High overlap = methods agree on WHICH words
# matter. Low overlap = they disagree fundamentally.
# We normalize tokens (lowercase, remove Ġ) so that RoBERTa
# subword tokens match LIME words fairly.
# ════════════════════════════════════════════════════════════

N_OVERLAP = 10

def top_n_words_set(scores: dict, n: int) -> set:
    if not scores:
        return set()
    top = sorted(scores.items(), key=lambda x: abs(x[1]), reverse=True)[:n]
    return {w.replace('Ġ', '').lower() for w, _ in top}

overlap_rows = []
for _, row in data.iterrows():
    wb_top   = top_n_words_set(row['wb'],   N_OVERLAP)
    gs_top   = top_n_words_set(row['gs'],   N_OVERLAP)
    lime_top = top_n_words_set(row['lime'], N_OVERLAP)
    overlap_rows.append({
        'question':    row['question'],
        'answer_type': row['answer_type'],
        'is_ai':       row['is_ai'],
        'wb_gs':       len(wb_top   & gs_top),
        'wb_lime':     len(wb_top   & lime_top),
        'gs_lime':     len(gs_top   & lime_top),
        'all_three':   len(wb_top   & gs_top & lime_top),
        'wb_top':      ', '.join(sorted(wb_top)),
        'gs_top':      ', '.join(sorted(gs_top)),
        'lime_top':    ', '.join(sorted(lime_top)),
    })

ov_df = pd.DataFrame(overlap_rows)
ov_df.to_csv(f'{OUT_DIR}/token_overlap_all.csv', index=False)
print(f"Saved: {OUT_DIR}/token_overlap_all.csv")

print(f"\nMean overlap (out of Top-{N_OVERLAP}):")
for pair, label in [('wb_gs','WB ∩ GS'),('wb_lime','WB ∩ LIME'),('gs_lime','GS ∩ LIME'),('all_three','All three')]:
    mean = ov_df[pair].mean()
    print(f"  {label}: {mean:.1f} words  ({mean/N_OVERLAP*100:.0f}%)")

pairs_plot  = ['wb_gs', 'wb_lime', 'gs_lime', 'all_three']
labels_plot = [f'WB ∩ GS', f'WB ∩ LIME', f'GS ∩ LIME', 'All three']
colors_plot = ['#2c6fad', '#8e44ad', '#b03a2e', '#555']

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.subplots_adjust(wspace=0.35)

# Left: boxplot across all 90 texts
bp = axes[0].boxplot([ov_df[p].values for p in pairs_plot],
                     labels=labels_plot, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 1.5},
                     whiskerprops={'linewidth': 0.8},
                     capprops={'linewidth': 0.8},
                     flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
for patch, c in zip(bp['boxes'], colors_plot):
    patch.set_facecolor(c); patch.set_alpha(0.65)
axes[0].set_ylabel(f'Shared words (out of {N_OVERLAP})')
axes[0].set_ylim(-0.5, N_OVERLAP + 0.5)
axes[0].axhline(N_OVERLAP, color='#aaa', linewidth=0.6, linestyle='--')

# Right: AI vs Human mean overlap
ai_ov  = ov_df[ov_df['is_ai']]
hum_ov = ov_df[~ov_df['is_ai']]
x = np.arange(len(pairs_plot))
w = 0.35
axes[1].bar(x - w/2, [ai_ov[p].mean()  for p in pairs_plot], w,
            label='AI texts',    color='#c0392b', alpha=0.8)
axes[1].bar(x + w/2, [hum_ov[p].mean() for p in pairs_plot], w,
            label='Human texts', color='#2c6fad', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels_plot)
axes[1].set_ylabel(f'Mean shared words (out of {N_OVERLAP})')
axes[1].set_ylim(0, N_OVERLAP)
axes[1].legend(fontsize=9)

plt.savefig(f'{OUT_DIR}/analysis_2_overlap.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/analysis_2_overlap.png', bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_DIR}/analysis_2_overlap.png")


# ════════════════════════════════════════════════════════════
# ANALYSIS 3 — CONFIDENCE: AI vs HUMAN TEXTS
#
# What: Extract the model confidence score from GradientShap
# (same model, same confidence as for WB and GS; LIME uses
# the same model so lime_conf is identical — we use gs_conf).
# Compare confidence on AI-generated vs human-written texts.
#
# Why useful: A well-calibrated detector should be more
# confident on AI texts (they tend to be more uniform) than
# on human texts (more varied style).
# ════════════════════════════════════════════════════════════

conf_df = data[['question', 'q_text', 'answer_type', 'is_ai', 'gs_conf', 'lime_conf']].copy()
conf_df.to_csv(f'{OUT_DIR}/confidence_all.csv', index=False)
print(f"Saved: {OUT_DIR}/confidence_all.csv")

ai_conf  = conf_df[conf_df['is_ai']]['gs_conf'].dropna()
hum_conf = conf_df[~conf_df['is_ai']]['gs_conf'].dropna()
print(f"\nAI  texts — mean: {ai_conf.mean():.4f}  std: {ai_conf.std():.4f}  min: {ai_conf.min():.4f}")
print(f"Hum texts — mean: {hum_conf.mean():.4f}  std: {hum_conf.std():.4f}  min: {hum_conf.min():.4f}")

# Heatmap data: question × answer_type
pivot = conf_df.pivot_table(index='question', columns='answer_type',
                             values='gs_conf', aggfunc='mean')
col_order = ['Gpt_Answer_1','Gpt_Answer_2','Gpt_Answer_3',
             'Human_Answer_1','Human_Answer_2','Human_Answer_3']
pivot = pivot.reindex(columns=col_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.subplots_adjust(wspace=0.4)

# Left: violin
vparts = axes[0].violinplot([ai_conf.values, hum_conf.values],
                             positions=[1, 2], showmedians=True, showextrema=True)
vparts['bodies'][0].set_facecolor('#c0392b'); vparts['bodies'][0].set_alpha(0.6)
vparts['bodies'][1].set_facecolor('#2c6fad'); vparts['bodies'][1].set_alpha(0.6)
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(['AI texts', 'Human texts'])
axes[0].set_ylabel('Confidence')
axes[0].set_ylim(0, 1.05)
# Add individual points
axes[0].scatter(np.full(len(ai_conf), 1)  + np.random.uniform(-0.05,0.05,len(ai_conf)),
                ai_conf.values,  s=15, alpha=0.4, color='#c0392b', zorder=3)
axes[0].scatter(np.full(len(hum_conf), 2) + np.random.uniform(-0.05,0.05,len(hum_conf)),
                hum_conf.values, s=15, alpha=0.4, color='#2c6fad', zorder=3)

# Right: heatmap
sns.heatmap(pivot, ax=axes[1], cmap='YlOrRd', vmin=0.5, vmax=1.0,
            annot=True, fmt='.2f', linewidths=0.4, linecolor='#ddd',
            cbar_kws={'label': 'Confidence', 'shrink': 0.8},
            xticklabels=['GPT 1','GPT 2','GPT 3','Human 1','Human 2','Human 3'])
axes[1].set_xlabel('')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=30, labelsize=8)
axes[1].tick_params(axis='y', labelsize=8)

plt.savefig(f'{OUT_DIR}/analysis_3_confidence.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/analysis_3_confidence.png', bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_DIR}/analysis_3_confidence.png")


# ════════════════════════════════════════════════════════════
# ANALYSIS 4 — SIGN AGREEMENT BETWEEN METHODS
#
# What: For every word that appears in two methods, check
# whether both give it a positive or both a negative score.
# We report the fraction of shared words where signs match.
#
# Why useful: Even if methods pick different specific words,
# do they at least agree on the DIRECTION of influence?
# Random baseline = 50%.  Above 50% = better than chance.
#
# Note: tokens with score 0.0 are excluded (LIME outputs
# exact zeros for words it couldn't distinguish).
# ════════════════════════════════════════════════════════════

def sign_agreement(a: dict, b: dict) -> float:
    shared = set(a.keys()) & set(b.keys())
    nonzero = [(a[w], b[w]) for w in shared if a[w] != 0 and b[w] != 0]
    if len(nonzero) < 3:
        return np.nan
    agree = sum(1 for av, bv in nonzero if np.sign(av) == np.sign(bv))
    return agree / len(nonzero)

sign_rows = []
for _, row in data.iterrows():
    wb_n   = normalize_keys(row['wb'])
    gs_n   = normalize_keys(row['gs'])
    lime_n = normalize_keys(row['lime'])
    sign_rows.append({
        'question':    row['question'],
        'answer_type': row['answer_type'],
        'is_ai':       row['is_ai'],
        'wb_gs':       sign_agreement(wb_n, gs_n),
        'wb_lime':     sign_agreement(wb_n, lime_n),
        'gs_lime':     sign_agreement(gs_n, lime_n),
    })

sign_df = pd.DataFrame(sign_rows)
sign_df.to_csv(f'{OUT_DIR}/sign_agreement_all.csv', index=False)
print(f"Saved: {OUT_DIR}/sign_agreement_all.csv")

pairs_s  = ['wb_gs', 'wb_lime', 'gs_lime']
labels_s = ['WB vs GS', 'WB vs LIME', 'GS vs LIME']
colors_s = ['#2c6fad', '#8e44ad', '#b03a2e']

print("\nMean sign agreement (random baseline = 50%):")
for p, l in zip(pairs_s, labels_s):
    print(f"  {l}: {sign_df[p].mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.subplots_adjust(wspace=0.35)

bp = axes[0].boxplot([sign_df[p].dropna().values for p in pairs_s],
                     labels=labels_s, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 1.5},
                     whiskerprops={'linewidth': 0.8},
                     capprops={'linewidth': 0.8},
                     flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
for patch, c in zip(bp['boxes'], colors_s):
    patch.set_facecolor(c); patch.set_alpha(0.65)
axes[0].axhline(0.5, color='#999', linewidth=0.8, linestyle='--', label='Random baseline (50%)')
axes[0].set_ylabel('Fraction with same sign')
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=8)

ai_sg  = sign_df[sign_df['is_ai']]
hum_sg = sign_df[~sign_df['is_ai']]
x = np.arange(3)
w = 0.35
axes[1].bar(x - w/2, [ai_sg[p].mean()  for p in pairs_s], w,
            label='AI texts',    color='#c0392b', alpha=0.8)
axes[1].bar(x + w/2, [hum_sg[p].mean() for p in pairs_s], w,
            label='Human texts', color='#2c6fad', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels_s)
axes[1].set_ylabel('Mean sign agreement')
axes[1].axhline(0.5, color='#999', linewidth=0.8, linestyle='--')
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=9)

plt.savefig(f'{OUT_DIR}/analysis_4_sign_agreement.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/analysis_4_sign_agreement.png', bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_DIR}/analysis_4_sign_agreement.png")


# ════════════════════════════════════════════════════════════
# ANALYSIS 5 — SCORE MAGNITUDE PER METHOD
#
# What: Compare the absolute size of attribution scores across
# methods. LIME scores are expected to be smaller because LIME
# fits a local linear model — its coefficients reflect marginal
# probability changes (~0–0.3), whereas gradient-based methods
# can accumulate larger values.
#
# We report: mean |score|, max |score|, and token count per text.
# Token count differs because WB/GS work on subword tokens
# while LIME works on full words.
# ════════════════════════════════════════════════════════════

mag_rows = []
for _, row in data.iterrows():
    for method, key in [('WhiteBox', 'wb'), ('GradientShap', 'gs'), ('LIME', 'lime')]:
        scores = row[key]
        if not scores:
            continue
        vals = list(scores.values())
        mag_rows.append({
            'question':    row['question'],
            'answer_type': row['answer_type'],
            'is_ai':       row['is_ai'],
            'method':      method,
            'mean_abs':    float(np.mean(np.abs(vals))),
            'max_abs':     float(np.max(np.abs(vals))),
            'n_tokens':    len(vals),
        })

mag_df = pd.DataFrame(mag_rows)
mag_df.to_csv(f'{OUT_DIR}/score_magnitude_all.csv', index=False)
print(f"Saved: {OUT_DIR}/score_magnitude_all.csv")

print("\nMean |score| per method:")
print(mag_df.groupby('method')['mean_abs'].describe().round(5).to_string())

METHOD_ORDER  = ['WhiteBox', 'GradientShap', 'LIME']
METHOD_COLORS = {'WhiteBox': '#2c6fad', 'GradientShap': '#2e8b57', 'LIME': '#b03a2e'}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
fig.subplots_adjust(wspace=0.4)

metrics = [('mean_abs', 'Mean |attribution score|'),
           ('max_abs',  'Max |attribution score|'),
           ('n_tokens', 'Token / word count per text')]

for ax, (metric, ylabel) in zip(axes, metrics):
    data_v = [mag_df[mag_df['method'] == m][metric].values for m in METHOD_ORDER]
    vp = ax.violinplot(data_v, positions=[1, 2, 3], showmedians=True, showextrema=True)
    for i, pc in enumerate(vp['bodies']):
        pc.set_facecolor(list(METHOD_COLORS.values())[i])
        pc.set_alpha(0.65)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(METHOD_ORDER, fontsize=9)
    ax.set_ylabel(ylabel)

plt.savefig(f'{OUT_DIR}/analysis_5_magnitude.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/analysis_5_magnitude.png', bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_DIR}/analysis_5_magnitude.png")


# ════════════════════════════════════════════════════════════
# ANALYSIS 6 — ATTRIBUTION SPREAD (CONCENTRATION RATIO)
#
# What: What fraction of the total |score| mass is carried by
# the Top-5 tokens? A ratio near 1.0 means the model focuses
# almost all attention on 5 words. A low ratio means the
# signal is spread across many tokens.
#
# Formula: sum(top-5 |scores|) / sum(all |scores|)
#
# Why useful: Tells us whether the explanation is
# interpretable (few decisive words) or diffuse (many
# tokens each contribute a little).
# ════════════════════════════════════════════════════════════

def concentration_ratio(scores: dict, top_k: int = 5) -> float:
    if not scores or len(scores) < top_k:
        return np.nan
    vals  = np.abs(list(scores.values()))
    total = vals.sum()
    if total == 0:
        return np.nan
    return np.sort(vals)[-top_k:].sum() / total

spread_rows = []
for _, row in data.iterrows():
    for method, key in [('WhiteBox', 'wb'), ('GradientShap', 'gs'), ('LIME', 'lime')]:
        c = concentration_ratio(row[key], top_k=5)
        if not np.isnan(c):
            spread_rows.append({
                'question':    row['question'],
                'answer_type': row['answer_type'],
                'is_ai':       row['is_ai'],
                'method':      method,
                'concentration': c,
            })

spread_df = pd.DataFrame(spread_rows)
spread_df.to_csv(f'{OUT_DIR}/concentration_all.csv', index=False)
print(f"Saved: {OUT_DIR}/concentration_all.csv")

print("\nConcentration ratio (Top-5 / Total |score|):")
print(spread_df.groupby('method')['concentration'].describe().round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.subplots_adjust(wspace=0.35)

data_c = [spread_df[spread_df['method'] == m]['concentration'].values for m in METHOD_ORDER]
bp = axes[0].boxplot(data_c, labels=METHOD_ORDER, patch_artist=True,
                     medianprops={'color': 'black', 'linewidth': 1.5},
                     whiskerprops={'linewidth': 0.8},
                     capprops={'linewidth': 0.8},
                     flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5})
for patch, c in zip(bp['boxes'], METHOD_COLORS.values()):
    patch.set_facecolor(c); patch.set_alpha(0.65)
axes[0].set_ylabel('Concentration ratio (Top-5 / Total)')
axes[0].set_ylim(0, 1.05)

ai_sp  = spread_df[spread_df['is_ai']]
hum_sp = spread_df[~spread_df['is_ai']]
x = np.arange(3)
w = 0.35
axes[1].bar(x - w/2, [ai_sp[ai_sp['method']  == m]['concentration'].mean() for m in METHOD_ORDER],
            w, label='AI texts',    color='#c0392b', alpha=0.8)
axes[1].bar(x + w/2, [hum_sp[hum_sp['method'] == m]['concentration'].mean() for m in METHOD_ORDER],
            w, label='Human texts', color='#2c6fad', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(METHOD_ORDER)
axes[1].set_ylabel('Mean concentration ratio')
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=9)

plt.savefig(f'{OUT_DIR}/analysis_6_spread.pdf', bbox_inches='tight')
plt.savefig(f'{OUT_DIR}/analysis_6_spread.png', bbox_inches='tight')
plt.show()
print(f"Saved: {OUT_DIR}/analysis_6_spread.png")


# ════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ════════════════════════════════════════════════════════════

summary = pd.DataFrame([
    ('AI texts — mean confidence',                  f"{ai_conf.mean():.4f}"),
    ('Human texts — mean confidence',               f"{hum_conf.mean():.4f}"),
    (f'Token overlap WB ∩ GS   (Top-{N_OVERLAP})', f"{ov_df['wb_gs'].mean():.1f} / {N_OVERLAP}"),
    (f'Token overlap WB ∩ LIME (Top-{N_OVERLAP})', f"{ov_df['wb_lime'].mean():.1f} / {N_OVERLAP}"),
    (f'Token overlap GS ∩ LIME (Top-{N_OVERLAP})', f"{ov_df['gs_lime'].mean():.1f} / {N_OVERLAP}"),
    ('Sign agreement WB vs GS',                     f"{sign_df['wb_gs'].mean():.2%}"),
    ('Sign agreement WB vs LIME',                   f"{sign_df['wb_lime'].mean():.2%}"),
    ('Sign agreement GS vs LIME',                   f"{sign_df['gs_lime'].mean():.2%}"),
    ('Mean |score| WhiteBox',                       f"{mag_df[mag_df['method']=='WhiteBox']['mean_abs'].mean():.5f}"),
    ('Mean |score| GradientShap',                   f"{mag_df[mag_df['method']=='GradientShap']['mean_abs'].mean():.5f}"),
    ('Mean |score| LIME',                           f"{mag_df[mag_df['method']=='LIME']['mean_abs'].mean():.5f}"),
    ('Concentration Top-5 WhiteBox',                f"{spread_df[spread_df['method']=='WhiteBox']['concentration'].mean():.2%}"),
    ('Concentration Top-5 GradientShap',            f"{spread_df[spread_df['method']=='GradientShap']['concentration'].mean():.2%}"),
    ('Concentration Top-5 LIME',                    f"{spread_df[spread_df['method']=='LIME']['concentration'].mean():.2%}"),
], columns=['Metric', 'Value'])

summary.to_csv(f'{OUT_DIR}/summary_table.csv', index=False)
print(f"\nSaved: {OUT_DIR}/summary_table.csv")
print("\n" + summary.to_string(index=False))

print(f"\n{'='*55}")
print("All outputs saved to:  ./{OUT_DIR}/")
print("  analysis_1_top_bottom.png   + .pdf")
print("  analysis_2_overlap.png      + .pdf")
print("  analysis_3_confidence.png   + .pdf")
print("  analysis_4_sign_agreement.png + .pdf")
print("  analysis_5_magnitude.png    + .pdf")
print("  analysis_6_spread.png       + .pdf")
print("  top_bottom_words_all.csv    (all Top/Bottom scores)")
print("  token_overlap_all.csv")
print("  confidence_all.csv")
print("  sign_agreement_all.csv")
print("  score_magnitude_all.csv")
print("  concentration_all.csv")
print("  summary_table.csv")